In [1]:
%pip install fuzzywuzzy[speedup]


Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix
from fuzzywuzzy import process


In [3]:
movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")


In [4]:
movie_data = pd.merge(ratings, movies, on='movieId')
movie_counts = movie_data['title'].value_counts()
popular_movies = movie_counts[movie_counts >= 50].index.tolist()

filtered_data = movie_data[movie_data['title'].isin(popular_movies)]
movie_user_matrix = filtered_data.pivot_table(index='title', columns='userId', values='rating').fillna(0)
ratings_sparse = csr_matrix(movie_user_matrix.values)


In [5]:
knn = NearestNeighbors(metric='cosine', algorithm='brute')
knn.fit(ratings_sparse)


,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [6]:
def search_movies(movies_df, query):
    query = query.strip()
    if query.isdigit() and len(query) == 4:
        year = query
        matched = movies_df[movies_df['title'].str.contains(f'\\({year}\\)', regex=True, case=False)]
    else:
        matched = movies_df[movies_df['title'].str.contains(query, case=False, na=False)]
    return matched[['title', 'genres']].drop_duplicates()


In [7]:
def recommend_similar_movies(movie_title, movie_matrix, sparse_matrix, model, movies_df, top_n=5):
    if movie_title not in movie_matrix.index:
        return f"Movie '{movie_title}' not found in dataset."

    idx = movie_matrix.index.get_loc(movie_title)
    distances, indices = model.kneighbors(sparse_matrix[idx], n_neighbors=top_n + 1)
    
    # Get genres of the base movie
    base_genres = movies_df[movies_df['title'] == movie_title]['genres'].values[0].split('|')
    
    recommendations = []
    for i in range(1, len(distances[0])):
        rec_title = movie_matrix.index[indices[0][i]]
        rec_genres = movies_df[movies_df['title'] == rec_title]['genres'].values[0].split('|')
        # Calculate genre overlap count
        genre_overlap = len(set(base_genres).intersection(set(rec_genres)))
        score = (1 - distances[0][i]) + (genre_overlap * 0.1)  # weight genre overlap lightly
        recommendations.append((rec_title, score))
    
    # Sort by combined score descending
    recommendations.sort(key=lambda x: x[1], reverse=True)
    return recommendations[:top_n]


In [8]:
query = input("Enter the movie: ").strip()
matched_movies = search_movies(movies, query)

if matched_movies.empty:
    print(f"No movies found for '{query}'.")
else:
    print(f"Found {len(matched_movies)} movies matching '{query}':\n")
    for title, genres in zip(matched_movies['title'], matched_movies['genres']):
        print(f"▶ {title} | Genres: {genres}")
        recs = recommend_similar_movies(title, movie_user_matrix, ratings_sparse, knn, movies, top_n=5)
        if isinstance(recs, str):
            print("  No recommendations found.\n")
        else:
            print("  Similar movies:")
            for rec_title, score in recs:
                print(f"    • {rec_title} (Score: {score:.3f})")
        print()


Found 22 movies matching 'batman':

▶ Batman Forever (1995) | Genres: Action|Adventure|Comedy|Crime
  Similar movies:
    • True Lies (1994) (Score: 0.943)
    • Batman (1989) (Score: 0.906)
    • Cliffhanger (1993) (Score: 0.802)
    • Ace Ventura: Pet Detective (1994) (Score: 0.720)
    • Dances with Wolves (1990) (Score: 0.686)

▶ Batman (1989) | Genres: Action|Crime|Thriller
  Similar movies:
    • Batman Forever (1995) (Score: 0.906)
    • True Lies (1994) (Score: 0.897)
    • Jurassic Park (1993) (Score: 0.839)
    • Terminator 2: Judgment Day (1991) (Score: 0.746)
    • Fugitive, The (1993) (Score: 0.740)

▶ Batman Returns (1992) | Genres: Action|Crime
  Similar movies:
    • RoboCop (1987) (Score: 0.735)
    • Die Hard 2 (1990) (Score: 0.601)
    • Starship Troopers (1997) (Score: 0.592)
    • Men in Black (a.k.a. MIB) (1997) (Score: 0.590)
    • Con Air (1997) (Score: 0.589)

▶ Batman & Robin (1997) | Genres: Action|Adventure|Fantasy|Thriller
  No recommendations found.


▶ Ba